In [1]:
import datetime
from concordia.agents import entity_agent
from concordia.agents import entity_agent_with_logging 
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.components.agent import concat_act_component
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.language_models.ollama import ollama_model
from concordia.environment.engines import asynchronous
from concordia.prefabs.simulation import generic as simulation
import concordia.prefabs.game_master as game_master_prefabs
import concordia.prefabs.entity as entity_prefabs
from concordia.typing import prefab as prefab_lib
from concordia.utils import async_measurements as async_measurements_lib
from concordia.utils import helper_functions
import pandas as pd
from IPython import display
import numpy as np
from numpy.linalg import norm
import sentence_transformers
import ollama
import json
import os
import yaml
import random
import itertools
import time

In [2]:
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

model = ollama_model.OllamaLanguageModel(
    model_name="deepseek-r1:8b",
)

In [3]:
with open('personas.yaml', 'r', encoding='utf-8') as f:
    yaml_data = yaml.safe_load(f)['personas']

In [4]:
class MpnetEmbedder:
    """Embedder using HuggingFace, adpted to Concordia."""
    
    def __init__(self, model):
        self._model = model
    def __call__(self, text: str) -> np.ndarray:
        return self._model.encode(text, show_progress_bar=False).astype(np.float32)
    def embed(self, text: str) -> np.ndarray:
        return self._model.encode(text, show_progress_bar=False).astype(np.float32)

    def embed_sentence(self, text: str) -> np.ndarray:
        return self.embed(text)

In [5]:
st_model = sentence_transformers.SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
embedder = MpnetEmbedder(st_model)

In [7]:
def load_mapping():
    with open("mfq_mapping.json") as f:
        mapping = json.load(f)

    return mapping

In [8]:
def analyze_results(results, mapping):

    analysis = []

    for r in results:

        scores = compute_mfq_scores(
            r["answers"],
            mapping
        )

        analysis.append({
            "persona": r["persona"],
            **scores
        })

    return analysis

In [9]:
def compute_mfq_scores(answers, mapping):

    scores = {}

    for foundation, questions in mapping.items():

        values = [] 

        for q in questions:
            if q in answers:
                values.append(answers[q])

        scores[foundation] = np.mean(values)

    return scores

In [10]:
with open("results.json", "r", encoding="utf-8") as f:
    json_results = json.load(f)

mapping = load_mapping()

analysis = analyze_results(json_results, mapping)

df_mft = pd.DataFrame(analysis)

df_mft = df_mft.dropna(subset=['care', 'fairness', 'loyalty', 'authority', 'purity'])

In [11]:
def build_concordia_agent(persona_yaml, mft_scores, current_news):
    name = persona_yaml['name']

    interests = "\n- ".join(persona_yaml['interests'])
    beliefs = "\n- ".join(persona_yaml['beliefs'])

    identity_prompt = f"""
You are {name}, {persona_yaml['age_range']} years old, from {persona_yaml['location']}.
Occupation: {persona_yaml['occupation']}
Description: {persona_yaml['small_description']}

Your main interests:
- {interests}

Your core beliefs:
- {beliefs}

Your Moral Foundations (0-5, higher = more important to you):
- Care/Harm: {mft_scores['care']:.2f}
- Fairness/Cheating: {mft_scores['fairness']:.2f}
- Loyalty/Betrayal: {mft_scores['loyalty']:.2f}
- Authority/Subversion: {mft_scores['authority']:.2f}
- Purity/Degradation: {mft_scores['purity']:.2f}

You are reading a news/political discussion forum. The trending topic
(already pinned as Post #0 in your feed) is:
"{current_news}"

How to act each turn (pick ONE action):
1. Read your visible posts AND replies. For each one, ask: does it activate
   any of MY high moral foundations (above), positively or negatively?
2. Choose the action that best matches your reaction:
   - share    : the post resonates strongly with your foundations and you
                want others to see it. Sharing amplifies the post for the
                whole forum. (Only top-level posts can be shared.)
   - upvote   : you agree with the post or reply.
   - downvote : the post or reply offends a foundation you score high on.
   - reply    : you want to argue, expand, or add nuance. You can reply to
                a post OR to another reply (a quote-reply, which gets
                attached to the same root post).
   - post     : ONLY if you have a genuinely new angle the existing posts
                do not cover. Avoid duplicating what others already said.
   - ignore   : the topic does not engage you. Lurking is realistic.

Output rules:
- Output EXACTLY ONE valid JSON object. No markdown fences, no prose.
- Use only ids that appear in your current feed. Never invent ids.
- Post ids and reply ids share the SAME number space; both appear in the
  visible ids list. Pass either one as the "post_id" field for upvote,
  downvote, and reply.
- share only works on top-level posts. Sharing a reply will be rejected.
- Stay on the trending topic. Do not talk about your daily routine.
- You can share each post only once. Do not share the same post again later.
- You can vote on each post or reply only once total: pick upvote OR downvote
  one time; do not vote on the same id again afterwards.
- If you already shared or voted on something, choose another action (reply,
  post) or ignore. Repeating a share or vote on the same id will be silently
  treated as ignore.

Action shapes:
{{"action": "share", "author": "{name}", "post_id": "<id>"}}
{{"action": "upvote", "author": "{name}", "post_id": "<id>"}}
{{"action": "downvote", "author": "{name}", "post_id": "<id>"}}
{{"action": "reply", "author": "{name}", "post_id": "<id>", "content": "..."}}
{{"action": "post", "author": "{name}", "title": "...", "content": "..."}}
{{"action": "ignore", "author": "{name}"}}
"""

    raw_memory_list = []
    memory = agent_components.memory.ListMemory(memory_bank=raw_memory_list)

    identity = agent_components.constant.Constant(state=identity_prompt)

    act_comp = concat_act_component.ConcatActComponent(model=model)

    agent = entity_agent_with_logging.EntityAgentWithLogging(
        agent_name=name,
        act_component=act_comp,
        context_components={
            'memory': memory,
            'identity': identity
        },
        measurements=async_measurements_lib.ReactiveMeasurements()
    )
    return agent

In [ ]:
num_batches = 10
batch_size = 5
max_steps = 20

MFT_KEYS = ['care', 'fairness', 'loyalty', 'authority', 'purity']

# Each topic carries a hand-coded MFT profile (0-5 scale) describing which
# foundations the news activates. Used later to compute moral resonance
# between the news and each persona.
NEWS_TOPICS = [
    {
        'id': 'hospital_donation',
        'title': "Children's Hospital Receives Record Donation and Expands Treatment for Rare Diseases",
        'body': "A nationwide campaign raised millions for a children's hospital, allowing the opening of new wings and free treatment for hundreds of vulnerable children.",
        'topic_mft': {'care': 5.0, 'fairness': 4.0, 'loyalty': 3.0, 'authority': 2.0, 'purity': 2.0},
    },
    {
        'id': 'congress_corruption',
        'title': "Congressman Uses Public Funds for Personal Trip and Faces Investigation",
        'body': "Documents indicate that resources allocated to the congressional office were used for personal expenses abroad.",
        'topic_mft': {'care': 2.0, 'fairness': 5.0, 'loyalty': 2.0, 'authority': 4.0, 'purity': 3.0},
    },
    {
        'id': 'school_anthem',
        'title': "School Reintroduces Daily National Anthem and Stricter Discipline Rules",
        'body': "The administration states that the measure aims to reinforce respect, order, and civic values among students.",
        'topic_mft': {'care': 1.0, 'fairness': 2.0, 'loyalty': 5.0, 'authority': 5.0, 'purity': 3.0},
    },
    {
        'id': 'religious_marriage',
        'title': "Religious Leader Criticizes School Campaign Supporting Same-Sex Marriage",
        'body': "During a public event, a religious leader stated that schools should prioritize academic content and preserve traditional family values.",
        'topic_mft': {'care': 2.0, 'fairness': 3.0, 'loyalty': 4.0, 'authority': 5.0, 'purity': 5.0},
    },
    {
        'id': 'drought_climate',
        'title': "Historic Drought Forces City to Ration Water and Suspend Classes",
        'body': "Reservoirs reached critical levels after months without rain. Experts linked the extreme conditions to the advance of climate change.",
        'topic_mft': {'care': 5.0, 'fairness': 4.0, 'loyalty': 2.0, 'authority': 2.0, 'purity': 3.0},
    },
    {
        'id': 'gun_law',
        'title': "Congress Approves Bill Tightening Rules for Gun Purchase and Ownership",
        'body': "Lawmakers approved new requirements for firearms acquisition, including training, psychological evaluations, and tracking measures. Supporters argue the policy could reduce deaths and accidents.",
        'topic_mft': {'care': 4.0, 'fairness': 4.0, 'loyalty': 3.0, 'authority': 4.0, 'purity': 2.0},
    },
]

# TCC §3.2: three forum titles (neutral / conservative / progressive).
# forum_mft is the moral profile each name signals to the agents.
FORUM_VARIANTS = [
    {'name': 'Rural News',        'forum_mft': {'care': 3.0, 'fairness': 3.0, 'loyalty': 3.0, 'authority': 3.0, 'purity': 3.0}},
    {'name': 'Voz Conservadora',  'forum_mft': {'care': 3.0, 'fairness': 3.0, 'loyalty': 4.5, 'authority': 4.5, 'purity': 4.5}},
    {'name': 'Voz Progressista',  'forum_mft': {'care': 4.5, 'fairness': 4.5, 'loyalty': 2.0, 'authority': 2.0, 'purity': 2.0}},
]

print(f"\nIniciando bateria de {num_batches} experimentos aleatórios (max_steps={max_steps})...")

base_dir = r"D:\breno\Documents\TCC\Resultados"
os.makedirs(base_dir, exist_ok=True)

current_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f.startswith("Experimento ")]
next_num = 8
if current_folders:
    numbers = []
    for folder in current_folders:
        try:
            numbers.append(int(folder.replace("Experimento ", "")))
        except ValueError:
            pass
    if numbers:
        next_num = max(max(numbers) + 1, 8)

new_folder = os.path.join(base_dir, f"Experimento - {next_num}")
os.makedirs(new_folder, exist_ok=True)
runs_folder = os.path.join(new_folder, 'runs')
os.makedirs(runs_folder, exist_ok=True)

run_id = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
all_actions_path = os.path.join(new_folder, 'all_actions.jsonl')
metadata_path = os.path.join(new_folder, 'metadata.json')
if os.path.exists(all_actions_path):
    os.remove(all_actions_path)

batch_metadata = {
    'run_id': run_id,
    'created_at': datetime.datetime.now().isoformat(),
    'num_batches': num_batches,
    'batch_size': batch_size,
    'max_steps': max_steps,
    'model_name': getattr(model, 'model_name', 'deepseek-r1:8b'),
    'results_folder': new_folder,
    'runs_folder': runs_folder,
    'all_actions_path': all_actions_path,
    'news_topics': NEWS_TOPICS,
    'forum_variants': FORUM_VARIANTS,
}
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(batch_metadata, f, ensure_ascii=False, indent=2)

print(f"\n📂 Diretório criado para salvar resultados: {new_folder}")
print(f"[RUN ID]: {run_id}")


def _mft_dict_to_vec(d):
    return np.array([float(d[k]) for k in MFT_KEYS], dtype=np.float64)


def _cos_sim(v1, v2):
    n1, n2 = norm(v1), norm(v2)
    if n1 == 0 or n2 == 0:
        return float('nan')
    return float(np.dot(v1, v2) / (n1 * n2))


def compute_summary(action_rows, forum_state, persona_mft_by_name, news_topic, forum_variant, group_mean_mft, group_similarity):
    """Compute diffusion metrics from the append-only action registry."""
    state = forum_state.get_state()
    posts = sorted(state['posts'].values(), key=lambda p: p['post_id'])
    if not posts:
        return {}

    seed = posts[0]
    derived = posts[1:]
    successful = [a for a in action_rows if a.get('status') == 'success']

    seed_actions = [a for a in successful if a.get('target_post_id') == 0]
    seed_replies = [a for a in seed_actions if a.get('action_type') == 'reply']
    seed_upvotes = [a for a in seed_actions if a.get('action_type') == 'upvote']
    seed_downvotes = [a for a in seed_actions if a.get('action_type') == 'downvote']
    seed_shares = [a for a in seed_actions if a.get('action_type') == 'share']

    seed_post = {
        'post_id': seed['post_id'],
        'author': seed['author'],
        'upvotes': len(seed_upvotes),
        'downvotes': len(seed_downvotes),
        'votes': len(seed_upvotes) - len(seed_downvotes),
        'shares': len(seed_shares),
        'reply_count': len(seed_replies),
        'unique_repliers': sorted({a['actor'] for a in seed_replies}),
        'unique_sharers': sorted({a['actor'] for a in seed_shares}),
    }

    status_counts = {}
    action_type_counts = {}
    for a in action_rows:
        status_counts[a.get('status', 'unknown')] = status_counts.get(a.get('status', 'unknown'), 0) + 1
        action_type_counts[a.get('action_type', 'unknown')] = action_type_counts.get(a.get('action_type', 'unknown'), 0) + 1

    # Derived engagement is action-level, not counter-level: successful actions
    # targeting non-seed posts plus successful new posts.
    derived_engagement_rows = [
        a for a in successful
        if (a.get('target_post_id') is not None and a.get('target_post_id') != 0)
        or (a.get('action_type') == 'post' and a.get('created_post_id') not in (None, 0))
    ]

    cascade_depth = 0
    if seed_actions:
        cascade_depth = 1
    if derived_engagement_rows:
        cascade_depth = 2

    actions_per_persona = {name: {'post': 0, 'reply': 0, 'share': 0,
                                   'upvote': 0, 'downvote': 0, 'ignore': 0,
                                   'duplicate_share': 0, 'duplicate_vote': 0,
                                   'reply_upvote': 0, 'reply_downvote': 0,
                                   'quote_reply': 0,
                                   'failed': 0}
                            for name in persona_mft_by_name}
    for a in action_rows:
        actor = a.get('actor')
        if actor not in actions_per_persona:
            continue
        effective = a.get('effective_action_type') or a.get('action_type')
        attempted = a.get('attempted_action_type') or a.get('action_type')
        counts_as_ignore = bool(a.get('counts_as_ignore'))
        duplicate = bool(a.get('duplicate_interaction'))
        target_kind = a.get('target_node_kind')
        if counts_as_ignore:
            actions_per_persona[actor]['ignore'] += 1
            if duplicate and attempted == 'share':
                actions_per_persona[actor]['duplicate_share'] += 1
            elif duplicate and attempted in ('upvote', 'downvote'):
                actions_per_persona[actor]['duplicate_vote'] += 1
            continue
        if a.get('status') == 'success' and effective in actions_per_persona[actor]:
            actions_per_persona[actor][effective] += 1
            if target_kind == 'reply':
                if effective == 'upvote':
                    actions_per_persona[actor]['reply_upvote'] += 1
                elif effective == 'downvote':
                    actions_per_persona[actor]['reply_downvote'] += 1
                elif effective == 'reply':
                    actions_per_persona[actor]['quote_reply'] += 1
        elif a.get('status') != 'success':
            actions_per_persona[actor]['failed'] += 1

    unique_engagers = sorted({
        a.get('actor') for a in successful
        if a.get('actor') in persona_mft_by_name
        and (a.get('effective_action_type') or a.get('action_type')) != 'ignore'
        and not a.get('counts_as_ignore')
    })

    duplicate_share_attempts = sum(
        1 for a in action_rows
        if a.get('duplicate_interaction')
        and (a.get('attempted_action_type') == 'share')
    )
    duplicate_vote_attempts = sum(
        1 for a in action_rows
        if a.get('duplicate_interaction')
        and a.get('attempted_action_type') in ('upvote', 'downvote')
    )
    counts_as_ignore_total = sum(
        1 for a in action_rows if a.get('counts_as_ignore')
    )
    consecutive_all_ignore_cycles_final = (
        forum_state.get_consecutive_all_ignore_cycles()
    )
    ignore_cycle_threshold = forum_state.get_ignore_cycle_threshold()
    terminated_by_ignore_cycles = (
        consecutive_all_ignore_cycles_final >= ignore_cycle_threshold
    )

    seed_thread_engagement_rows = [
        a for a in successful
        if a.get('target_root_post_id') == 0
        and (a.get('effective_action_type') or a.get('action_type')) != 'ignore'
        and not a.get('counts_as_ignore')
    ]
    reply_target_rows = [
        a for a in successful if a.get('target_node_kind') == 'reply'
    ]
    reply_vote_total = sum(
        1 for a in reply_target_rows
        if (a.get('effective_action_type') or a.get('action_type'))
        in ('upvote', 'downvote')
    )
    quote_reply_total = sum(
        1 for a in reply_target_rows
        if (a.get('effective_action_type') or a.get('action_type')) == 'reply'
    )
    not_shareable_attempts = sum(
        1 for a in action_rows if a.get('status') == 'not_shareable'
    )

    topic_vec = _mft_dict_to_vec(news_topic['topic_mft'])
    forum_vec = _mft_dict_to_vec(forum_variant['forum_mft'])

    persona_resonance = {}
    for name, vec in persona_mft_by_name.items():
        persona_resonance[name] = {
            'mft_vector': vec.tolist(),
            'topic_resonance': _cos_sim(vec, topic_vec),
            'forum_alignment': _cos_sim(vec, forum_vec),
            'actions': actions_per_persona[name],
        }

    return {
        'news_topic_id': news_topic['id'],
        'news_title': news_topic['title'],
        'topic_mft': news_topic['topic_mft'],
        'forum_variant': forum_variant['name'],
        'forum_mft': forum_variant['forum_mft'],
        'group_mean_mft': dict(zip(MFT_KEYS, [float(x) for x in group_mean_mft])),
        'group_cosine_similarity_pct': float(group_similarity),
        'seed_post': seed_post,
        'derived_post_count': len(derived),
        'derived_engagement_total': len(derived_engagement_rows),
        'cascade_depth': cascade_depth,
        'unique_engagers': unique_engagers,
        'unique_engagers_count': len(unique_engagers),
        'status_counts': status_counts,
        'action_type_counts': action_type_counts,
        'duplicate_share_attempts': duplicate_share_attempts,
        'duplicate_vote_attempts': duplicate_vote_attempts,
        'counts_as_ignore_total': counts_as_ignore_total,
        'consecutive_all_ignore_cycles_final': consecutive_all_ignore_cycles_final,
        'ignore_cycle_threshold': ignore_cycle_threshold,
        'terminated_by_ignore_cycles': terminated_by_ignore_cycles,
        'seed_thread_engagement_total': len(seed_thread_engagement_rows),
        'reply_vote_total': reply_vote_total,
        'quote_reply_total': quote_reply_total,
        'not_shareable_attempts': not_shareable_attempts,
        'persona_resonance': persona_resonance,
    }


persona_list = []
full_start_time = time.time()

for batch_index in range(num_batches):
    print(f"\n{'='*60}")
    print(f" EXPERIMENTO {batch_index + 1} DE {num_batches} ")
    print(f"{'='*60}")
    batch_start_time = time.time()

    news_topic = NEWS_TOPICS[batch_index % len(NEWS_TOPICS)]
    forum_variant = random.choice(FORUM_VARIANTS)
    forum_name = f"{forum_variant['name']} (Exp {batch_index + 1})"

    # Plain-text version used inside the persona prompt.
    current_news = f"{news_topic['title']} — {news_topic['body']}"

    print(f"\n[TEMA]: {news_topic['title']}")
    print(f"[FÓRUM]: {forum_name}")

    if len(persona_list) < batch_size:
        print("\n[SISTEMA] Embaralhando as 40 personas para garantir participação de todos...")
        persona_list = list(yaml_data)
        random.shuffle(persona_list)

    batch_group = persona_list[:batch_size]
    persona_list = persona_list[batch_size:]

    print("\n--- COMPOSIÇÃO DO GRUPO ---")
    active_agents = []
    vetores_mft_do_grupo = []
    persona_mft_by_name = {}
    persona_metadata = {}

    for persona_yaml in batch_group:
        name = persona_yaml['name']
        mft_row = df_mft[df_mft['persona'] == name]
        if mft_row.empty:
            print(f" -> AVISO: Dados de MFT não encontrados para {name}")
            continue
        mft_scores = mft_row.iloc[0]
        vetor_mft = mft_scores[MFT_KEYS].astype(float).values
        vetores_mft_do_grupo.append(vetor_mft)
        persona_mft_by_name[name] = vetor_mft
        persona_metadata[name] = {
            'actor_mft_care': float(vetor_mft[0]),
            'actor_mft_fairness': float(vetor_mft[1]),
            'actor_mft_loyalty': float(vetor_mft[2]),
            'actor_mft_authority': float(vetor_mft[3]),
            'actor_mft_purity': float(vetor_mft[4]),
            'actor_occupation': persona_yaml.get('occupation', ''),
            'actor_location': persona_yaml.get('location', ''),
            'actor_age_range': persona_yaml.get('age_range', ''),
        }
        print(f" -> {name:<25} | Vetor: [{', '.join(f'{v:.1f}' for v in vetor_mft)}]")
        active_agents.append(build_concordia_agent(persona_yaml, mft_scores, current_news))

    medias_grupo = np.zeros(len(MFT_KEYS))
    porcentagem_sim = float('nan')
    if len(vetores_mft_do_grupo) > 1:
        matriz_mft = np.array(vetores_mft_do_grupo)
        medias_grupo = np.mean(matriz_mft, axis=0)
        print("\n--- MÉDIA MORAL DO GRUPO (PERFIL DO FÓRUM) ---")
        print(f"Care: {medias_grupo[0]:.2f} | Fairness: {medias_grupo[1]:.2f} | "
              f"Loyalty: {medias_grupo[2]:.2f} | Authority: {medias_grupo[3]:.2f} | "
              f"Purity: {medias_grupo[4]:.2f}")

        sims = [_cos_sim(v1, v2) for v1, v2 in itertools.combinations(matriz_mft, 2)]
        sims = [s for s in sims if not np.isnan(s)]
        if sims:
            porcentagem_sim = float(np.mean(sims) * 100)
            tag = ""
            if porcentagem_sim > 94:
                tag = "  (Forte Bolha Ideológica)"
            elif porcentagem_sim < 82:
                tag = "  (Fórum Polarizado)"
            print(f"\n[MÉTRICA] Similaridade Vetorial do Grupo: {porcentagem_sim:.1f}%{tag}")

    print("\n--- INICIANDO FÓRUM ASSÍNCRONO ---")

    gm_memory_bank = basic_associative_memory.AssociativeMemoryBank(
        sentence_embedder=embedder
    )

    gm_prefab = game_master_prefabs.async_social_media.GameMaster(
        entities=active_agents,
        params={
            'name': 'forum_rules',
            'forum_name': forum_name,
            'seed_posts': [{
                'author': 'BREAKING NEWS',
                'title': news_topic['title'],
                'content': news_topic['body'],
            }],
            'ignore_cycle_threshold': 3,
            'measurements': async_measurements_lib.ReactiveMeasurements(),
        },
    )
    gm = gm_prefab.build(model=model, memory_bank=gm_memory_bank)

    sim_engine = asynchronous.Asynchronous()
    sim_engine.run_loop(
        game_masters=[gm],
        entities=active_agents,
        premise='',
        max_steps=max_steps,
        verbose=True,
    )

    print(f"\n--- FIM DO EXPERIMENTO {batch_index + 1} ---")

    forum_state = gm.get_component('__forum__')
    experiment_index = batch_index + 1
    run_prefix = f"run-{experiment_index:03d}"
    experiment_metadata = {
        'run_id': run_id,
        'experiment_index': experiment_index,
        'forum_name': forum_name,
        'forum_variant': forum_variant['name'],
        'news_topic_id': news_topic['id'],
        'news_title': news_topic['title'],
        'max_steps': max_steps,
        'batch_size': batch_size,
        'group_cosine_similarity_pct': porcentagem_sim,
    }

    actions_path = os.path.join(runs_folder, f"{run_prefix}-actions.jsonl")
    n_actions = forum_state.to_action_jsonl(
        actions_path,
        experiment_metadata=experiment_metadata,
        persona_metadata=persona_metadata,
    )
    forum_state.to_action_jsonl(
        all_actions_path,
        experiment_metadata=experiment_metadata,
        persona_metadata=persona_metadata,
        append=True,
    )
    action_rows = forum_state.get_action_log()
    print(f"💾 {n_actions} ações salvas em: {actions_path}")

    events_path = os.path.join(runs_folder, f"{run_prefix}-forum-snapshot.jsonl")
    n_events = forum_state.to_jsonl_events(events_path)
    print(f"💾 {n_events} eventos de estado salvos em: {events_path}")

    summary = compute_summary(
        action_rows=action_rows,
        forum_state=forum_state,
        persona_mft_by_name=persona_mft_by_name,
        news_topic=news_topic,
        forum_variant=forum_variant,
        group_mean_mft=medias_grupo,
        group_similarity=porcentagem_sim,
    )
    summary.update(experiment_metadata)
    summary_path = os.path.join(runs_folder, f"{run_prefix}-summary.json")
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"💾 Sumário salvo em: {summary_path}")

    if summary:
        sp = summary['seed_post']
        print(f"\n[DIFUSÃO] Post #0 → shares={sp['shares']}, votes={sp['votes']}, "
              f"replies={sp['reply_count']}, derived_posts={summary['derived_post_count']}, "
              f"engajamento_derivado={summary['derived_engagement_total']}, "
              f"cascade_depth={summary['cascade_depth']}, "
              f"unique_engagers={summary['unique_engagers_count']}")
        print(f"[INATIVIDADE] ciclos_all_ignore={summary['consecutive_all_ignore_cycles_final']}/"
              f"{summary['ignore_cycle_threshold']} "
              f"(terminado por inatividade: {summary['terminated_by_ignore_cycles']}), "
              f"duplicates_share={summary['duplicate_share_attempts']}, "
              f"duplicates_vote={summary['duplicate_vote_attempts']}, "
              f"counts_as_ignore={summary['counts_as_ignore_total']}")
        print(f"[REPLIES] seed_thread_engagement={summary['seed_thread_engagement_total']}, "
              f"reply_votes={summary['reply_vote_total']}, "
              f"quote_replies={summary['quote_reply_total']}, "
              f"not_shareable={summary['not_shareable_attempts']}")

    batch_end_time = time.time()
    min_b, sec_b = divmod(batch_end_time - batch_start_time, 60)
    print(f"\n[TEMPO] ⏱️ Experimento {batch_index + 1} levou {int(min_b)}m {int(sec_b)}s.")

full_end_time = time.time()
min_t, sec_t = divmod(full_end_time - full_start_time, 60)
print(f"\n{'='*60}")
print(f" SIMULAÇÃO COMPLETA FINALIZADA!")
print(f" ⏱️ Tempo total: {int(min_t)} minutos e {int(sec_t)} segundos.")
print(f" 📂 Resultados em: {new_folder}")
print(f"{'='*60}")


Iniciando bateria de 10 experimentos aleatórios (max_steps=20)...

📂 Diretório criado para salvar resultados: D:\breno\Documents\TCC\Resultados\Experimento - 33
[RUN ID]: 20260512-222033

 EXPERIMENTO 1 DE 10 

[TEMA]: Children's Hospital Receives Record Donation and Expands Treatment for Rare Diseases
[FÓRUM]: Rural News (Exp 1)

[SISTEMA] Embaralhando as 40 personas para garantir participação de todos...

--- COMPOSIÇÃO DO GRUPO ---
 -> Igor Almeida              | Vetor: [3.8, 3.3, 4.0, 1.8, 2.2]
 -> Fernanda Oliveira         | Vetor: [4.8, 4.3, 4.8, 3.5, 3.6]
 -> Marcos Ferreira           | Vetor: [4.2, 4.7, 1.8, 1.5, 1.8]
 -> André Luiz Barbosa        | Vetor: [3.5, 4.3, 2.2, 2.2, 1.6]
 -> Maria Aparecida Silva     | Vetor: [4.5, 3.2, 4.2, 4.2, 4.2]

--- MÉDIA MORAL DO GRUPO (PERFIL DO FÓRUM) ---
Care: 4.17 | Fairness: 3.97 | Loyalty: 3.40 | Authority: 2.65 | Purity: 2.68

[MÉTRICA] Similaridade Vetorial do Grupo: 94.5%  (Forte Bolha Ideológica)

--- INICIANDO FÓRUM ASSÍNCRONO ---

In [ ]:
# Aggregate action-level diffusion analysis across all experiments in `new_folder`.
# This cell treats all_actions.jsonl as the primary research dataset.

ANALYSIS_FOLDER = new_folder  # change to a fixed path to analyse a past run
all_actions_file = os.path.join(ANALYSIS_FOLDER, 'all_actions.jsonl')

if not os.path.exists(all_actions_file):
    raise FileNotFoundError(f"No action registry found at: {all_actions_file}")

df_actions = pd.read_json(all_actions_file, lines=True)
print(f"Carregadas {len(df_actions)} ações de {all_actions_file}\n")

if 'counts_as_ignore' not in df_actions.columns:
    df_actions['counts_as_ignore'] = False
if 'duplicate_interaction' not in df_actions.columns:
    df_actions['duplicate_interaction'] = False
if 'attempted_action_type' not in df_actions.columns:
    df_actions['attempted_action_type'] = df_actions['action_type']
if 'effective_action_type' not in df_actions.columns:
    df_actions['effective_action_type'] = df_actions['action_type']
if 'consecutive_all_ignore_cycles' not in df_actions.columns:
    df_actions['consecutive_all_ignore_cycles'] = 0
if 'target_node_kind' not in df_actions.columns:
    df_actions['target_node_kind'] = None
if 'target_root_post_id' not in df_actions.columns:
    df_actions['target_root_post_id'] = df_actions.get('target_post_id')
if 'parent_reply_id' not in df_actions.columns:
    df_actions['parent_reply_id'] = None
df_actions['counts_as_ignore'] = df_actions['counts_as_ignore'].fillna(False).astype(bool)
df_actions['duplicate_interaction'] = df_actions['duplicate_interaction'].fillna(False).astype(bool)
df_actions['target_root_post_id'] = df_actions['target_root_post_id'].where(
    df_actions['target_root_post_id'].notna(),
    df_actions['target_post_id'],
)

df_actions['is_success'] = df_actions['status'].eq('success')
df_actions['is_seed_action'] = df_actions['target_post_id'].eq(0)
df_actions['is_seed_thread'] = df_actions['target_root_post_id'].eq(0)
df_actions['is_reply_target'] = df_actions['target_node_kind'].eq('reply')
df_actions['is_quote_reply'] = (
    df_actions['parent_reply_id'].notna()
    & df_actions['effective_action_type'].eq('reply')
    & df_actions['is_success']
)
df_actions['is_effective_ignore'] = (
    df_actions['effective_action_type'].eq('ignore') | df_actions['counts_as_ignore']
)
df_actions['is_engagement'] = df_actions['is_success'] & ~df_actions['is_effective_ignore']
df_actions['is_real_failure'] = ~df_actions['is_success'] & ~df_actions['counts_as_ignore']


def _row_mft_vec(row):
    return np.array([
        row['actor_mft_care'],
        row['actor_mft_fairness'],
        row['actor_mft_loyalty'],
        row['actor_mft_authority'],
        row['actor_mft_purity'],
    ], dtype=np.float64)


topic_mft_by_id = {t['id']: _mft_dict_to_vec(t['topic_mft']) for t in NEWS_TOPICS}
forum_mft_by_name = {f['name']: _mft_dict_to_vec(f['forum_mft']) for f in FORUM_VARIANTS}

df_actions['topic_resonance'] = df_actions.apply(
    lambda row: _cos_sim(_row_mft_vec(row), topic_mft_by_id[row['news_topic_id']]),
    axis=1,
)
df_actions['forum_alignment'] = df_actions.apply(
    lambda row: _cos_sim(_row_mft_vec(row), forum_mft_by_name[row['forum_variant']]),
    axis=1,
)

print("=== Dataset principal ===")
display.display(df_actions.head())


print("\n=== 1) Distribuição de ações por tipo e status ===")
action_distribution = df_actions.groupby(['action_type', 'attempted_action_type', 'status']).size().reset_index(name='n')
display.display(action_distribution)

persona_action_distribution = df_actions.groupby(['actor', 'action_type', 'attempted_action_type', 'status']).size().reset_index(name='n')


print("\n=== 2) Taxa de compartilhamento por persona ===")
persona_rates = df_actions.groupby('actor').agg(
    total_actions=('action_seq', 'count'),
    successful_actions=('is_success', 'sum'),
    shares=('attempted_action_type', lambda s: (s == 'share').sum()),
    successful_shares=('attempted_action_type', lambda s: ((s == 'share') & df_actions.loc[s.index, 'is_success']).sum()),
    seed_shares=('attempted_action_type', lambda s: ((s == 'share') & df_actions.loc[s.index, 'is_success'] & df_actions.loc[s.index, 'is_seed_action']).sum()),
    replies=('attempted_action_type', lambda s: ((s == 'reply') & df_actions.loc[s.index, 'is_success']).sum()),
    posts=('attempted_action_type', lambda s: ((s == 'post') & df_actions.loc[s.index, 'is_success']).sum()),
    duplicate_share_attempts=('duplicate_interaction', lambda s: (s & df_actions.loc[s.index, 'attempted_action_type'].eq('share')).sum()),
    duplicate_vote_attempts=('duplicate_interaction', lambda s: (s & df_actions.loc[s.index, 'attempted_action_type'].isin(['upvote', 'downvote'])).sum()),
    counts_as_ignore=('counts_as_ignore', 'sum'),
    effective_ignores=('is_effective_ignore', 'sum'),
    failed_actions=('is_real_failure', 'sum'),
    avg_topic_resonance=('topic_resonance', 'mean'),
    avg_forum_alignment=('forum_alignment', 'mean'),
).reset_index()
persona_rates['share_rate'] = persona_rates['successful_shares'] / persona_rates['total_actions'].replace(0, np.nan)
persona_rates['seed_share_rate'] = persona_rates['seed_shares'] / persona_rates['total_actions'].replace(0, np.nan)
display.display(persona_rates.sort_values('successful_shares', ascending=False))


print("\n=== 3) Difusão do post-semente por tópico e fórum ===")
seed_diffusion = df_actions[df_actions['is_seed_action'] & df_actions['is_success']].groupby(
    ['news_topic_id', 'forum_variant', 'action_type']
).size().reset_index(name='n')
display.display(seed_diffusion)

seed_summary = df_actions[df_actions['is_seed_action'] & df_actions['is_success']].groupby(
    ['news_topic_id', 'forum_variant']
).agg(
    seed_actions=('action_seq', 'count'),
    seed_shares=('action_type', lambda s: (s == 'share').sum()),
    seed_replies=('action_type', lambda s: (s == 'reply').sum()),
    seed_upvotes=('action_type', lambda s: (s == 'upvote').sum()),
    seed_downvotes=('action_type', lambda s: (s == 'downvote').sum()),
    unique_engagers=('actor', 'nunique'),
).reset_index()
display.display(seed_summary)


print("\n=== 4) Taxa de erro por persona (excluindo duplicatas) ===")
error_rates = df_actions.groupby('actor').agg(
    total_actions=('action_seq', 'count'),
    failed_actions=('is_real_failure', 'sum'),
    duplicate_attempts=('duplicate_interaction', 'sum'),
).reset_index()
error_rates['error_rate'] = error_rates['failed_actions'] / error_rates['total_actions'].replace(0, np.nan)
display.display(error_rates.sort_values('error_rate', ascending=False))

status_by_topic = df_actions.groupby(['news_topic_id', 'status']).size().reset_index(name='n')


print("\n=== 4.1) Tentativas duplicadas (share/voto) por persona ===")
duplicate_breakdown = df_actions[df_actions['duplicate_interaction']].groupby(
    ['actor', 'attempted_action_type']
).size().reset_index(name='n')
display.display(duplicate_breakdown)


print("\n=== 4.2) Encerramento por inatividade (ciclos all-ignore) ===")
if 'experiment_index' in df_actions.columns:
    termination_summary = df_actions.groupby(
        ['experiment_index', 'forum_variant', 'news_topic_id']
    ).agg(
        total_actions=('action_seq', 'count'),
        counts_as_ignore_total=('counts_as_ignore', 'sum'),
        duplicate_share_attempts=('duplicate_interaction', lambda s: (s & df_actions.loc[s.index, 'attempted_action_type'].eq('share')).sum()),
        duplicate_vote_attempts=('duplicate_interaction', lambda s: (s & df_actions.loc[s.index, 'attempted_action_type'].isin(['upvote', 'downvote'])).sum()),
        consecutive_all_ignore_cycles_final=('consecutive_all_ignore_cycles', 'max'),
    ).reset_index()
    termination_summary['terminated_by_ignore_cycles'] = (
        termination_summary['consecutive_all_ignore_cycles_final'] >= 3
    )
    display.display(termination_summary)
else:
    termination_summary = pd.DataFrame()
    print("(coluna experiment_index ausente — dataset antigo)")


print("\n=== 5) Ressonância MFT vs engajamento ===")
engagement_by_actor_topic = df_actions.groupby(['actor', 'news_topic_id', 'forum_variant']).agg(
    actions=('action_seq', 'count'),
    engagements=('is_engagement', 'sum'),
    shares=('action_type', lambda s: ((s == 'share') & df_actions.loc[s.index, 'is_success']).sum()),
    topic_resonance=('topic_resonance', 'mean'),
    forum_alignment=('forum_alignment', 'mean'),
).reset_index()

corr_cols = ['topic_resonance', 'forum_alignment', 'actions', 'engagements', 'shares']
correlations = engagement_by_actor_topic[corr_cols].corr()
print("Correlação com engajamento/compartilhamento:")
display.display(correlations.loc[['topic_resonance', 'forum_alignment'], ['engagements', 'shares']].round(3))


print("\n=== 6) Compartilhamento por faixa de fundamento moral ===")
def _bin(v):
    if v < 2.5:
        return 'low'
    if v < 3.75:
        return 'mid'
    return 'high'

mft_propensity_tables = []
for foundation, column in {
    'care': 'actor_mft_care',
    'fairness': 'actor_mft_fairness',
    'loyalty': 'actor_mft_loyalty',
    'authority': 'actor_mft_authority',
    'purity': 'actor_mft_purity',
}.items():
    tmp = df_actions.copy()
    tmp[f'{foundation}_bin'] = tmp[column].map(_bin)
    table = tmp.groupby(f'{foundation}_bin').agg(
        actions=('action_seq', 'count'),
        engagements=('is_engagement', 'sum'),
        shares=('action_type', lambda s: ((s == 'share') & tmp.loc[s.index, 'is_success']).sum()),
    ).reset_index()
    table['foundation'] = foundation
    table['share_rate'] = table['shares'] / table['actions'].replace(0, np.nan)
    mft_propensity_tables.append(table)
    print(f"\n-- {foundation.upper()} --")
    display.display(table)

mft_propensity = pd.concat(mft_propensity_tables, ignore_index=True)


print("\n=== 7) Engajamento em respostas (reply-as-target) ===")
reply_actions = df_actions[df_actions['is_reply_target']]
if not reply_actions.empty:
    reply_engagement = reply_actions.groupby('actor').agg(
        reply_upvotes=('effective_action_type', lambda s: ((s == 'upvote') & reply_actions.loc[s.index, 'is_success']).sum()),
        reply_downvotes=('effective_action_type', lambda s: ((s == 'downvote') & reply_actions.loc[s.index, 'is_success']).sum()),
        quote_replies=('is_quote_reply', 'sum'),
        not_shareable_attempts=('status', lambda s: (s == 'not_shareable').sum()),
    ).reset_index()
    display.display(reply_engagement.sort_values(
        ['quote_replies', 'reply_upvotes', 'reply_downvotes'], ascending=False
    ))
else:
    reply_engagement = pd.DataFrame(
        columns=['actor', 'reply_upvotes', 'reply_downvotes', 'quote_replies',
                 'not_shareable_attempts']
    )
    print("(nenhuma ação direcionada a respostas no dataset)")


print("\n=== 8) Estrutura de threads por experimento ===")
if 'experiment_index' in df_actions.columns:
    successful_replies = df_actions[
        df_actions['is_success']
        & df_actions['effective_action_type'].eq('reply')
    ]
    thread_structure = successful_replies.groupby(
        ['experiment_index', 'forum_variant', 'news_topic_id']
    ).agg(
        total_replies=('action_seq', 'count'),
        top_level_replies=('parent_reply_id', lambda s: s.isna().sum()),
        quote_replies=('parent_reply_id', lambda s: s.notna().sum()),
        unique_repliers=('actor', 'nunique'),
    ).reset_index()
    thread_structure['quote_reply_share'] = (
        thread_structure['quote_replies']
        / thread_structure['total_replies'].replace(0, np.nan)
    )
    display.display(thread_structure)
else:
    thread_structure = pd.DataFrame()
    print("(coluna experiment_index ausente — dataset antigo)")


print("\n=== 9) Engajamento na thread do post-semente (root_post_id == 0) ===")
seed_thread_engagement = df_actions[
    df_actions['is_seed_thread']
    & df_actions['is_engagement']
].groupby(['actor', 'effective_action_type', 'target_node_kind']).size().reset_index(name='n')
display.display(seed_thread_engagement)


agg_dir = os.path.join(ANALYSIS_FOLDER, 'aggregated')
os.makedirs(agg_dir, exist_ok=True)
df_actions.to_csv(os.path.join(agg_dir, 'all_actions.csv'), index=False)
action_distribution.to_csv(os.path.join(agg_dir, 'action_distribution.csv'), index=False)
persona_action_distribution.to_csv(os.path.join(agg_dir, 'persona_action_distribution.csv'), index=False)
persona_rates.to_csv(os.path.join(agg_dir, 'persona_rates.csv'), index=False)
seed_diffusion.to_csv(os.path.join(agg_dir, 'seed_diffusion.csv'), index=False)
seed_summary.to_csv(os.path.join(agg_dir, 'seed_summary.csv'), index=False)
error_rates.to_csv(os.path.join(agg_dir, 'error_rates.csv'), index=False)
status_by_topic.to_csv(os.path.join(agg_dir, 'status_by_topic.csv'), index=False)
engagement_by_actor_topic.to_csv(os.path.join(agg_dir, 'engagement_by_actor_topic.csv'), index=False)
mft_propensity.to_csv(os.path.join(agg_dir, 'mft_propensity.csv'), index=False)
correlations.to_csv(os.path.join(agg_dir, 'correlations.csv'))
duplicate_breakdown.to_csv(os.path.join(agg_dir, 'duplicate_breakdown.csv'), index=False)
if not termination_summary.empty:
    termination_summary.to_csv(os.path.join(agg_dir, 'termination_summary.csv'), index=False)
reply_engagement.to_csv(os.path.join(agg_dir, 'reply_engagement.csv'), index=False)
if not thread_structure.empty:
    thread_structure.to_csv(os.path.join(agg_dir, 'thread_structure.csv'), index=False)
seed_thread_engagement.to_csv(os.path.join(agg_dir, 'seed_thread_engagement.csv'), index=False)
print(f"\n💾 Tabelas agregadas salvas em: {agg_dir}")

In [13]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Oliver Cromwell',
            'goal': 'become lord protector',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'King Charles I',
            'goal': 'avoid execution for treason',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            'extra_event_resolution_steps': '',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': [
                'The king was captured by Parliamentary forces in 1646.',
                'Charles I was tried for treason and found guilty.',
            ],
        },
    ),
]

In [14]:
import inspect
assinatura = inspect.signature(asynchronous.Asynchronous.__init__)
print(f"O Motor Assíncrono exige estes parâmetros: {assinatura}")

O Motor Assíncrono exige estes parâmetros: (self, call_to_make_observation: str = 'What is the current situation faced by {name}? What do they now observe? Only include information of which they are aware.', call_to_next_acting: str = 'Which entities act next?', call_to_next_action_spec: str = 'In what action spec format should {name} respond? Respond in  one of the provided formats and use no additional words.', call_to_resolve: str = 'Because of all that came before, what happens next?', call_to_check_termination: str = 'Is the game/simulation finished?', call_to_next_game_master: str = 'Which rule set should we use for the next step?', sleep_time: float = 0.1)
